In [8]:
#Import packages
using Pkg, CSV, DataFrames, Statistics, Plots, Ipopt, Combinatorics, Distances, LinearAlgebra, AmplNLWriter, NBInclude, Gurobi, JuMP, Graphs, GraphRecipes#, PyCall

In [9]:
@nbinclude("Similarity Factor & Bid-Ask Prices Parameterized.ipynb")
@nbinclude("Cost Function Parameterized.ipynb")

costfunct (generic function with 1 method)

In [ ]:

#set tabu list length for moving window
tabu_list = [[5,5],[5,5],[5,5],[5,5],[5,5],[5,5],[5,5],[5,5],[5,5],[5,5],[5,5],[5,5],[5,5],[5,5],[5,5],[5,5],[5,5],[5,5],[5,5],[5,5]]

20-element Vector{Vector{Int64}}:
 [5, 5]
 [5, 5]
 [5, 5]
 [5, 5]
 [5, 5]
 [5, 5]
 [5, 5]
 [5, 5]
 [5, 5]
 [5, 5]
 [5, 5]
 [5, 5]
 [5, 5]
 [5, 5]
 [5, 5]
 [5, 5]
 [5, 5]
 [5, 5]
 [5, 5]
 [5, 5]

In [11]:
#Compute forbidden subtours and return list of edges to forbid. Parameters are a list of components (in this case, cycles) for each callback solution, and a collection of callback edges.
function forbidden_tours(componentlist, cb_edges)
    #Initialize empty container to add subtour components
    component_container = []
    for component in componentlist
        #Indicator variable for component that includes null node; if null node is present, do not forbid. Otherwise, forbid the path.
        includes_null = 0
        #if the length of the component is one (if there is no edge/cycle), do not forbid
        if length(component) <= 1
            continue
        #if the length of a component is two (the cycle includes only two nodes such as 5 -> 3 -> 5), forbid it.
        elseif length(component) == 2
            push!(component_container, component)
            continue
        else
            #if the length of the component is greater than two and includes 5, this is an appropriate cycle that includes the null node. Do not forbid it.
            for elmt in component
                if elmt == 5
                        includes_null = 1
                end
            end
        end
        #if the length of the component is greater than two and does not include the null node, then forbid it because it represents a subtour. Recall that we are also forbidding all cycles that have a length of two (2 edges)
        if includes_null != 1
            push!(component_container, component)
        end
    end
    
    #Initialize empty container to store forbidden edges in order of their component.
    edge_container = []
    #for forbidden components in the callback solution, compute the relevant edges for each component and add them to the edge_container.
    for component in component_container
        #for each component, initialize a container to store edges for the component.
        edge_set = []
        #for each element within the component, find the relevant edge by searching for the element within edge source nodes.
        for elmt in component
            for edge in cb_edges
                if src(edge) == elmt #|| dst(edge) == elmt
                        push!(edge_set,(src(edge),dst(edge)))
                end
            end
        end
        #push the edge set for the component to the broader container
        push!(edge_container, edge_set)
    end
    #Return the edge container
    return edge_container
end

forbidden_tours (generic function with 1 method)

In [12]:
tabu_index = 1

function tabu_list_push(soln_matrix,tabu_list,tabu_length)
    global tabu_index
    global tabu_list
    #push!(tabu_list, [col_num[1],row_num[1]])

    #Find solution pair
    dummy_col = soln_matrix[5,:]
    dummy_row = soln_matrix[:,5]
    
    col_num = findall(dummy_col->dummy_col==1, dummy_col)
    row_num = findall(dummy_row->dummy_row==1, dummy_row)

    #revolving index
    if (tabu_index <= tabu_length)
        tabu_list[tabu_index] = [col_num[1],row_num[1]]
        tabu_index += 1
    else
        tabu_list[1] = [col_num[1],row_num[1]]
        tabu_index = 2
    end
    return tabu_list
end

function TSP_Pairs_Trade(similarity, ask_price_df, bid_price_df,tabu_length)
    
    global tabu_index
    global tabu_list

    #Initialize our model:
    pairs_trading_model = Model(Gurobi.Optimizer)
    
    index_max = size(similarity)[1]
    
    @variable(pairs_trading_model, x[i= 1:(index_max+1), j=1:(index_max+1)], Bin)
    @objective(pairs_trading_model, Min, costfunct(x, similarity, ask_price_df, bid_price_df))
    
    #inflow, outflow, equality
    @constraint(pairs_trading_model, inflow[i in 1:index_max], sum(x[i,:]) <= 1)
    @constraint(pairs_trading_model, outflow[j in 1:index_max], sum(x[:,j]) <= 1)
    @constraint(pairs_trading_model, equality[z in 1:(index_max+1)], sum(x[:,z]) - sum(x[z,:]) == 0)
    
    #dummy constraint
    @constraint(pairs_trading_model, dummyin, sum(x[(index_max+1),:]) == 1)
    @constraint(pairs_trading_model, dummyout, sum(x[:,(index_max+1)]) == 1)
    
    #Constraint that trades must be profitable (cost function < 0)
    #Idea: Modulo Arithmetic (counter that iterates through each element of the list and loops back to the start)
    
    @constraint(pairs_trading_model, tabulist[i in 1:size(tabu_list)[1]], x[5,tabu_list[i][1]] + x[tabu_list[i][2],5] <= 1)

    #Lazy constraint to eliminate subtours and short cycles of length two from the solution when they arise.
    function subtour_elimination_callback(cb_data)
        status = callback_node_status(cb_data, pairs_trading_model)
        if status != MOI.CALLBACK_NODE_STATUS_INTEGER
            return  # Only run at integer solutions
        end
        
        #Convert callback solution in matrix form to a directed graph
        cb_graph = Graphs.DiGraph(callback_value.(cb_data, pairs_trading_model[:x]))
        #Assign a list of the graph components (in this case, cycles) to the componentlist variable
        componentlist = strongly_connected_components(cb_graph)
        #Store edges of the directed graph in a collection variable cb_edges
        cb_edges = collect(edges(cb_graph))
        #display(callback_value.(cb_data, pairs_trading_model[:x]))
        #print("This is a callback solution")
        #call the forbidden_tours function to locate forbidden cycles and return the relevant edges to forbid for each one
        edge_container = forbidden_tours(componentlist, cb_edges)
        #If the function returns nothing, then do not initialize any lazy constraint
        if length(edge_container) == 0
            return
        else
            #display(edge_container)
            
            #For each forbidden cycle, build a lazy constraint to forbid the relevant edges by ensuring the sum of edges is less than the length of the component, effectively breaking the cycle.
            for term in edge_container
                edge_limit = length(term)
                #display(edge_limit)
                #display(term)
                con = @build_constraint(sum(pairs_trading_model[:x][edge[1], edge[2]] for edge in term) <= edge_limit-1)
                #display(con)
                MOI.submit(pairs_trading_model, MOI.LazyConstraint(cb_data), con)
            end
        end    
        return
    end

    set_attribute(
        pairs_trading_model,
        MOI.LazyConstraintCallback(),
        subtour_elimination_callback,
    )

    #Optimize model
    optimize!(pairs_trading_model)

    #Build solution matrix
    soln_matrix = round.(Int, value.(x))

    #Add solution to tabu list
    tabu_list = tabu_list_push(soln_matrix,tabu_list,tabu_length)

    display(value.(x))
    #print("this is a valid solution")
    
    println(objective_value(pairs_trading_model))
    println("this is the obj value")
    
    if objective_value(pairs_trading_model) > 0
        #return
    #else
        return value.(x)
    end

end



#for each in tabu_list
   #@constraint( x[i,j] + x[z,i] <=1)

#lazy constraints
#save model redundancies

TSP_Pairs_Trade (generic function with 1 method)

In [ ]:
@time TSP_Pairs_Trade(similarity, ask_price_df, bid_price_df,10)

5×5 Matrix{Float64}:
 -0.0  -0.0  -0.0   1.0  -0.0
 -0.0  -0.0  -0.0  -0.0   0.0
 -0.0   0.0  -0.0   0.0  -0.0
 -0.0   0.0  -0.0  -0.0   1.0
  1.0  -0.0   0.0  -0.0   0.0

Set parameter Username
Set parameter LicenseID to value 2604272
Academic license - for non-commercial use only - expires 2025-12-30
Set parameter LazyConstraints to value 1
Gurobi Optimizer version 12.0.0 build v12.0.0rc1 (mac64[arm] - Darwin 24.1.0 24B2082)

CPU model: Apple M4 Pro
Thread count: 12 physical cores, 12 logical processors, using up to 12 threads

Non-default parameters:
LazyConstraints  1

Optimize a model with 35 rows, 25 columns and 110 nonzeros
Model fingerprint: 0x57358d33
Variable types: 0 continuous, 25 integer (25 binary)
Coefficient statistics:
  Matrix range     [1e+00, 2e+00]
  Objective range  [5e+04, 4e+10]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e+00, 1e+00]
         Consider reformulating model or setting NumericFocus parameter
         to avoid numerical issues.
Found heuristic solution: objective 4.161279e+10
Presolve removed 21 rows and 1 columns
Presolve time: 0.00s
Presolved: 14 rows, 24 columns, 80 nonzeros
Variable types: 0 continuous,

In [15]:
for i in 1:3
     TSP_Pairs_Trade(similarity, ask_price_df, bid_price_df,10)
end

5×5 Matrix{Float64}:
 -0.0  -0.0  -0.0   1.0  -0.0
 -0.0  -0.0  -0.0  -0.0   1.0
 -0.0   0.0  -0.0   0.0  -0.0
 -0.0   1.0  -0.0  -0.0   0.0
  1.0  -0.0   0.0  -0.0   0.0

5×5 Matrix{Float64}:
 -0.0  -0.0  -0.0   1.0  -0.0
 -0.0  -0.0   1.0  -0.0   0.0
 -0.0  -0.0  -0.0   0.0   1.0
 -0.0   1.0  -0.0  -0.0   0.0
  1.0  -0.0   0.0  -0.0   0.0

5×5 Matrix{Float64}:
 -0.0  -0.0  -0.0   1.0  -0.0
 -0.0  -0.0  -0.0  -0.0  -0.0
  1.0   0.0  -0.0   0.0  -0.0
 -0.0   0.0  -0.0  -0.0   1.0
  0.0   0.0   1.0  -0.0   0.0

Set parameter Username
Set parameter LicenseID to value 2604272
Academic license - for non-commercial use only - expires 2025-12-30
Set parameter LazyConstraints to value 1
Gurobi Optimizer version 12.0.0 build v12.0.0rc1 (mac64[arm] - Darwin 24.1.0 24B2082)

CPU model: Apple M4 Pro
Thread count: 12 physical cores, 12 logical processors, using up to 12 threads

Non-default parameters:
LazyConstraints  1

Optimize a model with 35 rows, 25 columns and 111 nonzeros
Model fingerprint: 0x1843b110
Variable types: 0 continuous, 25 integer (25 binary)
Coefficient statistics:
  Matrix range     [1e+00, 2e+00]
  Objective range  [5e+04, 4e+10]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e+00, 1e+00]
         Consider reformulating model or setting NumericFocus parameter
         to avoid numerical issues.
Found heuristic solution: objective 4.161279e+10
Presolve removed 20 rows and 1 columns
Presolve time: 0.00s
Presolved: 15 rows, 24 columns, 83 nonzeros
Variable types: 0 continuous,